In [5]:
from google.colab import userdata
API_key = userdata.get('gemini')

In [7]:
!pip install -U langchain langchain-community -q
!pip install -U langchain-google-genai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.0 MB/s eta 0:00:00


In [8]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI

In [9]:
llm_model = "gemini-2.5-flash"

In [17]:
os.environ["GOOGLE_API_KEY"] = API_key
model = ChatGoogleGenerativeAI(
    model=llm_model,
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

In [18]:
response = model.invoke("What's the capital of the Moon?")

response

AIMessage(content="The Moon does not have a capital.\n\nIt's not a country or a political entity, and there are no permanent human settlements or governments on its surface. It's a natural satellite of Earth.\n\nIf humanity were to establish permanent colonies on the Moon in the future, they might eventually designate a primary administrative center, but that's purely hypothetical at this point!", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019cb848-4ad2-7cc3-ae94-50aaea663002-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 10, 'output_tokens': 622, 'total_tokens': 632, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 546}})

In [19]:
print(response.content)

The Moon does not have a capital.

It's not a country or a political entity, and there are no permanent human settlements or governments on its surface. It's a natural satellite of Earth.

If humanity were to establish permanent colonies on the Moon in the future, they might eventually designate a primary administrative center, but that's purely hypothetical at this point!


In [20]:
from pprint import pprint

pprint(response.response_metadata)

{'finish_reason': 'STOP',
 'model_name': 'gemini-2.5-flash',
 'model_provider': 'google_genai',
 'safety_ratings': []}


In [16]:
print(response.usage_metadata)

{'input_tokens': 10, 'output_tokens': 622, 'total_tokens': 632, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 546}}


[ChatGoogleGenerativeAI integration](https://docs.langchain.com/oss/python/integrations/chat/google_generative_ai#image-generation)

# Initialising and invoking an agent

In [21]:
from langchain.agents import create_agent

In [23]:
agent = create_agent(model=model)

In [24]:
from langchain.messages import HumanMessage

In [26]:
response = agent.invoke(
      {"messages" : [HumanMessage(content="What's the capital of the Moon?")]}
    )

In [31]:
pprint(response)

{'messages': [HumanMessage(content="What's the capital of the Moon?", additional_kwargs={}, response_metadata={}, id='25082cb2-eea3-4590-aa8d-54a43579346d'),
              AIMessage(content="The Moon does not have a capital.\n\nIt's not a country or a political entity, and there are no permanent human settlements or governments on its surface. It's a natural satellite of Earth.\n\nIf humanity were to establish permanent colonies on the Moon in the future, they might eventually designate a primary administrative center, but that's purely hypothetical at this point!", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019cb84d-deb2-7573-aab3-41c14dcc8099-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 10, 'output_tokens': 622, 'total_tokens': 632, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 546}})]}


In [32]:
print(response["messages"][-1].content)

The Moon does not have a capital.

It's not a country or a political entity, and there are no permanent human settlements or governments on its surface. It's a natural satellite of Earth.

If humanity were to establish permanent colonies on the Moon in the future, they might eventually designate a primary administrative center, but that's purely hypothetical at this point!


In [33]:
# Context or memory
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What's the capital of the Moon?"),
    AIMessage(content="The capital of the Moon is Luna City."),
    HumanMessage(content="Interesting, tell me more about Luna City")]})


In [34]:
pprint(response)

{'messages': [HumanMessage(content="What's the capital of the Moon?", additional_kwargs={}, response_metadata={}, id='76e17d2e-c772-4e39-985b-bd5382e2eb79'),
              AIMessage(content='The capital of the Moon is Luna City.', additional_kwargs={}, response_metadata={}, id='b311ae87-8a8a-4c5e-8a81-36f1bf0ceb00', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content='Interesting, tell me more about Luna City', additional_kwargs={}, response_metadata={}, id='123c9436-da35-4bb1-9a5f-ed8967ab6ef9'),
              AIMessage(content='My apologies! I need to clarify something important from my previous answer.\n\nThe Moon does not actually have a capital city, and therefore no "Luna City" in reality. I made an error in stating that it did.\n\nThe concept of "Luna City" is a popular one in science fiction, literature, and future speculative plans for lunar colonization. It represents humanity\'s aspirations to establish permanent settlements beyond Earth.\n\nIn these fi

# Streaming Output
- to reduce latency

In [35]:
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="Tell me all about Luna City, the capital of the Moon")]},
    stream_mode="messages"
):

    # token is a message chunk with token content
    # metadata contains which node produced the token

    if token.content:  # Check if there's actual content
        print(token.content, end="", flush=True)  # Print token

Welcome to Luna City, the bustling, resilient, and ever-expanding capital of humanity's first off-world colony!

**Official Designation:** Selenopolis Prime, though universally known as Luna City.
**Location:** Primarily situated within and around the rim of a large, ancient crater (often imagined as near the lunar poles for access to water ice, or in a historically significant region like Mare Tranquillitatis for symbolic reasons). For this description, let's place it in a vast, permanently shadowed crater near the South Pole, offering both protection and vital resources.

---

## All About Luna City: The Capital of the Moon

Luna City is more than just a settlement; it's a testament to human ingenuity, a vibrant melting pot of cultures, and the undisputed hub of all lunar activity. It serves as the administrative, scientific, industrial, and cultural heart of the Moon.

### 1. Founding and History

*   **Early 21st Century:** Initial robotic probes and small human missions confirm si

# system prompt

In [39]:
# Basic prompting
system_prompt = "You are a science fiction writer, create a capital city at the users request."

In [40]:
# Few-shot examples
system_prompt = """

You are a science fiction writer, create a space capital city at the users request.

User: What is the capital of mars?
Scifi Writer: Marsialis

User: What is the capital of Venus?
Scifi Writer: Venusovia

"""

In [41]:
# Structured prompts
system_prompt = """

You are a science fiction writer, create a space capital city at the users request.

Please keep to the below structure.

Name: The name of the capital city

Location: Where it is based

Vibe: 2-3 words to describe its vibe

Economy: Main industries

"""

In [42]:
question = HumanMessage(content="What's the capital of the moon?")

scifi_agent = create_agent(
    model=model,
    system_prompt=system_prompt
)

response = scifi_agent.invoke(
    {"messages": [question]}
)

print(response['messages'][1].content)

Name: Luna Prime

Location: Within the permanently shadowed regions of Shackleton Crater, Lunar South Pole.

Vibe: Resilient, Pioneering, Elegant

Economy: Helium-3 Fusion Fuel Extraction, Microgravity Manufacturing, Astroscience Research, Interstellar Port Services


# Anther why using outpus Schema

In [43]:
from pydantic import BaseModel

In [44]:
class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

In [45]:
question = HumanMessage(content="What's the capital of the moon?")

scifi_agent = create_agent(
    model=model,
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
    response_format=CapitalInfo
)

response = scifi_agent.invoke(
    {"messages": [question]}
)

print(response['messages'][1].content)

{"name": "Luna Prime", "location": "Mare Tranquillitatis, Earth's Moon", "vibe": "Bustling, high-tech, and aspirational, with a constant hum of industry and innovation beneath a vast, star-dusted dome.", "economy": "Driven by advanced lunar resource extraction, zero-G manufacturing of specialized components, and astrotourism."}


In [47]:
pprint(response['messages'])

[HumanMessage(content="What's the capital of the moon?", additional_kwargs={}, response_metadata={}, id='761cb61c-3a80-4202-b542-10fdafaa2f62'),
 AIMessage(content='{"name": "Luna Prime", "location": "Mare Tranquillitatis, Earth\'s Moon", "vibe": "Bustling, high-tech, and aspirational, with a constant hum of industry and innovation beneath a vast, star-dusted dome.", "economy": "Driven by advanced lunar resource extraction, zero-G manufacturing of specialized components, and astrotourism."}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019cb863-71c9-7383-a7c4-a5648f43aff3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 26, 'output_tokens': 260, 'total_tokens': 286, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 179}})]


In [48]:
response["structured_response"]

CapitalInfo(name='Luna Prime', location="Mare Tranquillitatis, Earth's Moon", vibe='Bustling, high-tech, and aspirational, with a constant hum of industry and innovation beneath a vast, star-dusted dome.', economy='Driven by advanced lunar resource extraction, zero-G manufacturing of specialized components, and astrotourism.')

In [49]:
response["structured_response"].name

'Luna Prime'